# 🚀 Pipeline Huấn luyện & Đánh giá I_MMSeg (Chia lại tập theo chuẩn SCAR: 243 Train / 61 Val / 76 Test)
> **Nghiên cứu & Đối sánh Benchmark:** Phân đoạn tổn thương cơ tim đa phương thái (**bSSFP/CINE, LGE, T2w**) trên tập dữ liệu **MyoPS-380**.
> **Kiến trúc:** **I_MMSeg** (Interactive Multimodal Medical Segmentation with Text Prompts & Cross-Attention).

---

### 📌 Các điểm cải tiến trọng tâm trong Pipeline này:
1. **Chia lại tập huấn luyện chuẩn xác 100% y chang SCAR:**
   - **Tập Train (243 bệnh nhân / 1578 lát cắt 2D):** Huấn luyện mô hình, loại bỏ nguy cơ data leakage.
   - **Tập Validation (61 bệnh nhân / 394 lát cắt 2D, 61 volume 3D):** Kiểm định mô hình sau mỗi epoch, tự động theo dõi và lưu `best.pth`.
   - **Tập Test (76 bệnh nhân / 76 volume 3D):** Kiểm thử độc lập hoàn toàn, đối sánh trực tiếp và công bằng với SCAR.
2. **Cơ chế Validation & Auto Best Checkpointing:**
   - Đánh giá định kỳ theo epoch trên tập Validation không dùng data augmentation ngẫu nhiên.
   - Tự động lưu `best.pth` (khi Mean Dice đạt đỉnh mới), lưu `latest.pth` và xuất file `metrics.csv`.
3. **Đo tốc độ Inference toàn diện (Inference Latency & Throughput Benchmark):**
   - ⏱️ **Từng lát cắt (Per-slice):** Đo chính xác thời gian forward pass (ms/slice), min, max, median, P95.
   - ⏱️ **Từng ca bệnh (Per-case):** Đo tổng thời gian xử lý từng ca 3D (s/case), số slice, FPS từng ca, kèm điểm số Dice/HD95.
   - ⏱️ **Tổng thời gian toàn bộ tập Test:** Đo tổng thời gian chạy hết 76 ca kiểm thử, tính Throughput (slices/sec) và xuất bảng báo cáo `per_case_speed_and_metrics.csv` & `test_speed_summary.json`.

## 🖥️ Bước 1: Kiểm tra GPU & Kết nối Google Drive
Đảm bảo bạn đã kích hoạt **GPU** trong Colab (`Runtime` -> `Change runtime type` -> Chọn `T4 GPU` hoặc `A100` / `L4`).

In [ ]:
# 1. Kiểm tra cấu hình GPU và VRAM hiện có
!nvidia-smi

# 2. Kết nối Google Drive để đồng bộ checkpoints, dataset và weights
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Đã kết nối Google Drive thành công!")
except ImportError:
    print("ℹ️ Đang chạy trên môi trường cục bộ (Local Environment).")

## 📥 Bước 2: Clone Repository I_MMSeg Mới Nhất
Clone mã nguồn I_MMSeg (đã tích hợp chia tập SCAR và đo tốc độ inference) từ GitHub:
`https://github.com/Duc247/I_MMSegI_HaveVlid`

In [ ]:
# Thiết lập thư mục làm việc cho Colab hoặc Local
import os
from pathlib import Path

if os.path.exists("/content"):
    project_root = Path("/content/I_MMSegI_HaveVlid")
    if not project_root.exists():
        # Kiểm tra nếu đã có thư mục repo trên Google Drive (Colab Notebooks/NCKH_V2)
        drive_repo_candidates = [
            Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/I_MMSegI_HaveVlid"),
            Path("/content/drive/MyDrive/NCKH_V2/I_MMSegI_HaveVlid"),
        ]
        found_in_drive = False
        for drive_src in drive_repo_candidates:
            if drive_src.exists():
                print(f"Sao chép mã nguồn từ Drive: {drive_src}")
                !cp -r "$drive_src" /content/I_MMSegI_HaveVlid
                found_in_drive = True
                break
        if not found_in_drive:
            print("Đang clone repository từ GitHub: https://github.com/Duc247/I_MMSegI_HaveVlid.git...")
            !git clone https://github.com/Duc247/I_MMSegI_HaveVlid.git /content/I_MMSegI_HaveVlid
    %cd /content/I_MMSegI_HaveVlid
else:
    print("Thư mục hiện tại:", os.getcwd())

print("Thư mục làm việc hiện tại:", os.getcwd())
!ls -lh

## 📦 Bước 3: Cài đặt các thư viện cần thiết
Cài đặt các gói phụ thuộc: `torch`, `torchvision`, `open-clip-torch`, `medpy`, `SimpleITK`, `tensorboardX`, `pandas`, `matplotlib`, `seaborn`.

In [ ]:
# Cài đặt requirements và các gói benchmark cần thiết
!pip install -q ml_collections open-clip-torch medpy SimpleITK tensorboardX pandas matplotlib seaborn h5py scipy tqdm

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("Device Count:", torch.cuda.device_count())
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")
    print(f"Memory Reserved:  {torch.cuda.memory_reserved(0)/(1024**2):.1f} MB")

## 🧠 Bước 4: Chuẩn bị Pretrained Weights (`R50-ViT-B_16.npz`) & Text Embeddings
Trọng số ResNetV2 ImageNet-21k được nạp đồng bộ vào cả 3 nhánh encoder (bSSFP, LGE, T2w) để tối ưu khả năng trích xuất đặc trưng.

In [ ]:
# Tạo thư mục chứa pretrained weights
!mkdir -p model/vit_checkpoint/imagenet21k

local_vit_path = Path("model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz")

# Nếu file hiện tại bị hỏng hoặc nhỏ hơn 10MB (file rỗng do lỗi tải trước đó), xóa để nạp lại
if local_vit_path.exists() and local_vit_path.stat().st_size < 10 * 1024 * 1024:
    print(f"⚠️ Phát hiện file weights bị hỏng/rỗng ({local_vit_path.stat().st_size} bytes), đang xóa để nạp file chuẩn...")
    local_vit_path.unlink()

# Danh sách đường dẫn ViT weights trên Google Drive theo cấu trúc Drive thực tế của bạn:
drive_vit_candidates = [
    Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/ViT_checkpoint/R50-ViT-B_16.npz"),
    Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/MyoPS380/ViT_checkpoint/R50-ViT-B_16.npz"),
    Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/R50-ViT-B_16.npz"),
    Path("/content/drive/MyDrive/google_drive/ViT_checkpoint/R50-ViT-B_16.npz"),
    Path("/content/drive/MyDrive/NCKH_V2/i_mmseg/ViT_checkpoint/R50-ViT-B_16.npz"),
    Path("/content/drive/MyDrive/NCKH_V2/ViT_checkpoint/R50-ViT-B_16.npz"),
]

found_weights = False
if not local_vit_path.exists() or local_vit_path.stat().st_size < 100 * 1024 * 1024:
    for cand in drive_vit_candidates:
        if cand.exists() and cand.stat().st_size > 100 * 1024 * 1024:
            print(f"✅ Tìm thấy ViT weights trên Google Drive: {cand} ({cand.stat().st_size / (1024*1024):.1f} MB)")
            print(f"Đang sao chép vào {local_vit_path}...")
            !cp "$cand" "$local_vit_path"
            found_weights = True
            break

    # Nếu chưa có trên Drive, tự động tải trực tiếp từ Google Storage chính thức (~440MB)
    if not local_vit_path.exists() or local_vit_path.stat().st_size < 100 * 1024 * 1024:
        print("Tải R50-ViT-B_16.npz từ Google ViT official storage (~440MB)...")
        print("(Lưu ý: URL chính thức có dấu '+' giữa R50 và ViT)")
        !wget -c -O "$local_vit_path" https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz

if local_vit_path.exists() and local_vit_path.stat().st_size > 100 * 1024 * 1024:
    print(f"✅ Pretrained weights sẵn sàng: {local_vit_path} ({local_vit_path.stat().st_size / (1024*1024):.1f} MB)")
else:
    print("⚠️ Chưa tìm thấy weights hợp lệ. Mô hình sẽ được khởi tạo ngẫu nhiên nếu không truyền weights.")

# Kiểm tra Text Features Embeddings của BiomedCLIP
text_feat_dir = Path("text_features")
text_feat_dir.mkdir(parents=True, exist_ok=True)
p1 = text_feat_dir / "embedding_class_information.pth"
p2 = text_feat_dir / "embedding_MRI_information.pth"

if p1.exists() and p2.exists():
    print("✅ Text features embeddings đã sẵn sàng trong thư mục text_features/")
else:
    print("Đang tạo text embeddings từ BiomedCLIP...")
    from train import Biomedclip_model
    Biomedclip_model()
    print("✅ Đã sinh text embeddings thành công!")

## 🗂️ Bước 5: Chuẩn bị Dataset MyoPS380
Đồng bộ thư mục `Processed_data` chứa 3 chuỗi xung (**bSSFP**, **LGE**, **T2w**) từ Google Drive hoặc thư mục cục bộ.

In [ ]:
# Chuẩn bị thư mục dữ liệu MyoPS380
import os
from pathlib import Path

target_dataset_dir = Path("MyoPS380_dataset/Processed_data")

# Danh sách các vị trí dataset trên Google Drive theo cấu trúc Drive thực tế của bạn:
drive_data_candidates = [
    Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/MyoPS380/Processed_data"),
    Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/MyoPS380"),
    Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/MyoPS380/MyoPS380_dataset/Processed_data"),
    Path("/content/drive/MyDrive/NCKH_V2/MyoPS380/Processed_data"),
    Path("/content/drive/MyDrive/NCKH_V2/i_mmseg/MyoPS380_dataset/Processed_data"),
    Path("/content/drive/MyDrive/google_drive/MyoPS380_dataset/Processed_data"),
]

if not (target_dataset_dir / "bSSFP" / "train_npz").exists():
    found_data = False
    for cand in drive_data_candidates:
        if (cand / "bSSFP" / "train_npz").exists():
            actual_src = cand
            print(f"✅ Tìm thấy Processed_data tại Drive: {actual_src}")
            !mkdir -p MyoPS380_dataset
            !rm -rf MyoPS380_dataset/Processed_data
            !ln -s "$actual_src" "MyoPS380_dataset/Processed_data" || cp -r "$actual_src" "MyoPS380_dataset/Processed_data"
            found_data = True
            break
        elif (cand / "Processed_data" / "bSSFP" / "train_npz").exists():
            actual_src = cand / "Processed_data"
            print(f"✅ Tìm thấy Processed_data tại Drive: {actual_src}")
            !mkdir -p MyoPS380_dataset
            !rm -rf MyoPS380_dataset/Processed_data
            !ln -s "$actual_src" "MyoPS380_dataset/Processed_data" || cp -r "$actual_src" "MyoPS380_dataset/Processed_data"
            found_data = True
            break

    if not found_data:
        # Kiểm tra nếu có file zip trong thư mục Colab Notebooks/NCKH_V2
        zip_candidates = [
            Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/MyoPS++_roi128mm_76cases.zip"),
            Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/MyoPS++_roi160mm_to128_80cases.zip"),
            Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/MyoPS380.zip"),
        ]
        for z in zip_candidates:
            if z.exists():
                print(f"📦 Phát hiện file zip tại: {z}")
                print(f"Để giải nén file này, chạy: !unzip -q -o "{z}" -d MyoPS380_dataset/Processed_data/")
                break

    if not found_data:
        print("⚠️ Chưa tìm thấy Processed_data. Hãy kiểm tra lối tắt MyoPS380 trong Colab Notebooks/NCKH_V2.")

# Kiểm tra dữ liệu 3 modalities
for mod in ["bSSFP", "LGE", "T2w"]:
    train_dir = target_dataset_dir / mod / "train_npz"
    test_dir = target_dataset_dir / mod / "test_vol_h5"
    if train_dir.exists() and test_dir.exists():
        n_train = len(list(train_dir.glob("*.npz")))
        n_test = len(list(test_dir.glob("*.h5")))
        print(f"✅ {mod:6s} -> train_npz: {n_train:4d} files | test_vol_h5: {n_test:3d} files")
    else:
        print(f"⚠️ Chưa tìm thấy đủ dữ liệu cho modality: {mod} ({train_dir})")

## ✂️ Bước 6: Đồng bộ Phân chia Dữ liệu SCAR (243 Train / 61 Val / 76 Test)
Nhánh SCAR chia 380 ca bệnh chuẩn hóa như sau:
- **Tập Test cố định (`test_vol.txt`):** 76 bệnh nhân (không bao giờ dùng để train hoặc chọn mô hình).
- **304 bệnh nhân còn lại (seed=1234, tỉ lệ val=0.2):**
  - **Tập Train (`train.txt`):** 243 bệnh nhân (**1578 lát cắt 2D**).
  - **Tập Validation (`val.txt` & `val_vol.txt`):** 61 bệnh nhân (**394 lát cắt 2D** / **61 volumes 3D**).

> 💡 **Khác biệt quan trọng:** Bản gốc I_MMSeg gộp chung cả 304 bệnh nhân (1972 lát cắt) vào `train.txt` mà không có tập val. Ở đây ta chia lại đúng chuẩn SCAR để **đảm bảo tính công bằng khoa học tuyệt đối**.

In [ ]:
# Xác thực và hiển thị thông tin phân chia dữ liệu chuẩn SCAR
import json
from pathlib import Path

list_dir = Path("list")
list_dir.mkdir(parents=True, exist_ok=True)

# Kiểm tra các file split
splits_info = {}
for split_name in ["train", "val", "val_vol", "test_vol", "all"]:
    split_file = list_dir / f"{split_name}.txt"
    if split_file.exists():
        lines = [l.strip() for l in split_file.read_text(encoding="utf-8").splitlines() if l.strip()]
        splits_info[split_name] = lines
        print(f"📄 {split_file.name:15s}: {len(lines):5d} dòng")
    else:
        print(f"❌ Thiếu file: {split_file.name}")

# Kiểm tra tính toàn vẹn (Data Leakage Check)
if "train" in splits_info and "val" in splits_info and "test_vol" in splits_info:
    train_slices = set(splits_info["train"])
    val_slices = set(splits_info["val"])
    test_cases = set(splits_info["test_vol"])

    overlap_train_val = train_slices & val_slices
    print("\n🔍 Kiểm tra trùng lặp Train và Val slices:", len(overlap_train_val))
    assert len(overlap_train_val) == 0, "LỖI: Trùng lặp giữa Train và Val!"
    print("✅ Hoàn hảo: Train và Val hoàn toàn tách biệt!")
    print(f"Tổng số slice: Train ({len(train_slices)}) + Val ({len(val_slices)}) = {len(train_slices) + len(val_slices)} slices.")

# Đọc metadata nếu có
meta_file = list_dir / "split_metadata.json"
if meta_file.exists():
    print("\nMetadata chia tập:")
    print(meta_file.read_text(encoding="utf-8"))

## 🏋️ Bước 7: Huấn luyện Mô hình I_MMSeg với Split SCAR & Auto Best Checkpointing
Mô hình sẽ được huấn luyện trên **1578 lát cắt của 243 ca bệnh** và được đánh giá trên **394 lát cắt của 61 ca bệnh Validation** sau mỗi epoch:
- Tự động theo dõi `val_loss`, `val_dice_myo`, `val_dice_scar`, `val_dice_edema`, `val_mean_dice`.
- Khi Validation Mean Dice đạt kỷ lục mới -> Tự động lưu checkpoint `best.pth`!
- Checkpoint `latest.pth` và toàn bộ lịch sử được ghi vào file `metrics.csv`.

In [ ]:
# Cấu hình huấn luyện I_MMSeg chuẩn SCAR split
output_run_dir = "runs/I_MMSeg_SCAR_Split"

# Đường dẫn backup chuẩn xác trên Google Drive (Colab Notebooks/NCKH_V2)
if os.path.exists("/content/drive/MyDrive/Colab Notebooks/NCKH_V2"):
    backup_drive_dir = "/content/drive/MyDrive/Colab Notebooks/NCKH_V2/I_MMSeg_runs"
elif os.path.exists("/content/drive/MyDrive/NCKH_V2"):
    backup_drive_dir = "/content/drive/MyDrive/NCKH_V2/I_MMSeg_runs"
else:
    backup_drive_dir = "/content/drive/MyDrive/I_MMSeg_runs"

# Tạo thư mục runs & backup Drive
!mkdir -p "$output_run_dir"
!mkdir -p "$backup_drive_dir"

# Lệnh huấn luyện I_MMSeg
!python train.py \
    --dataset Myops \
    --list_dir list \
    --img_size 128 \
    --batch_size 24 \
    --n_gpu 1 \
    --base_lr 0.001 \
    --max_epochs 300 \
    --val_interval 1 \
    --vit_name R50-ViT-B_16 \
    --vit_patches_size 16 \
    --n_skip 3 \
    --pretrained_path model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz \
    --output_dir "$output_run_dir" \
    --backup_dir "$backup_drive_dir" \
    --deterministic 1 \
    --seed 1234

## ⏱️ Bước 8: Đánh giá Toàn Diện & Đo Tốc Độ Inference trên Toàn Bộ Tập Test (76 Ca Bệnh)
> **TÍNH NĂNG ĐO LƯỜNG TỐC ĐỘ THEO YÊU CẦU:**
> 1. ⏱️ **Từng lát cắt (Per-slice):** Đo chính xác latency chạy forward pass GPU cho từng lát cắt ($ms/	ext{slice}$).
> 2. ⏱️ **Từng ca bệnh (Per-case):** Đo tổng thời gian xử lý từng ca 3D ($s/	ext{case}$), số lát cắt, thời gian trung bình lát cắt của ca, FPS từng ca, kèm điểm số Dice/HD95.
> 3. ⏱️ **Tổng thời gian toàn bộ tập Test:** Đo tổng thời gian chạy hết 76 ca kiểm thử, tính Throughput ($FPS$) và xuất file `per_case_speed_and_metrics.csv` & `test_speed_summary.json`.

In [ ]:
# Đánh giá checkpoint tốt nhất (best.pth) trên toàn bộ 76 ca kiểm thử test_vol
best_checkpoint = "runs/I_MMSeg_SCAR_Split/best.pth"
if not os.path.exists(best_checkpoint):
    # Fallback nếu dùng epoch checkpoint
    best_checkpoint = "runs/I_MMSeg_SCAR_Split/latest.pth"

eval_output_dir = "runs/I_MMSeg_SCAR_Split/evaluation_speed"
!mkdir -p "$eval_output_dir"

print(f"Đang sử dụng checkpoint: {best_checkpoint}")

!python evaluate_speed.py \
    --checkpoint "$best_checkpoint" \
    --data_root MyoPS380_dataset/Processed_data \
    --list_dir list \
    --output_dir "$eval_output_dir" \
    --img_size 128 \
    --vit_name R50-ViT-B_16 \
    --save_predictions \
    --warmup_cases 2

## 📊 Bước 9: Trực quan hóa Biểu đồ Huấn luyện & Tốc độ Inference
Vẽ biểu đồ Loss/Dice của quá trình huấn luyện và biểu đồ phân phối thời gian inference (ms/slice, s/case, FPS).

In [ ]:
# 1. Trực quan hóa đường cong huấn luyện Train Loss / Val Loss & Val Dice
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
metrics_csv = Path("runs/I_MMSeg_SCAR_Split/metrics.csv")

if metrics_csv.is_file():
    df_metrics = pd.read_csv(metrics_csv)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Biểu đồ Loss
    axes[0].plot(df_metrics["epoch"], df_metrics["train_loss"], label="Train Loss", color="royalblue", lw=2)
    if "val_loss" in df_metrics.columns and (df_metrics["val_loss"] > 0).any():
        axes[0].plot(df_metrics["epoch"], df_metrics["val_loss"], label="Val Loss", color="crimson", lw=2)
    axes[0].set_title("Loss qua các Epoch (Train vs Val)", fontsize=14, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    # Biểu đồ Dice Score
    if "dice_mean" in df_metrics.columns and (df_metrics["dice_mean"] > 0).any():
        axes[1].plot(df_metrics["epoch"], df_metrics["dice_myo"], label="Myocardium", color="forestgreen", lw=1.8)
        axes[1].plot(df_metrics["epoch"], df_metrics["dice_scar"], label="Scar (Sẹo)", color="firebrick", lw=2)
        axes[1].plot(df_metrics["epoch"], df_metrics["dice_edema"], label="Edema (Phù)", color="darkorange", lw=1.8)
        axes[1].plot(df_metrics["epoch"], df_metrics["dice_mean"], label="Mean Dice", color="purple", lw=2.5, linestyle="--")
        axes[1].set_title("Chỉ số Validation Dice qua các Epoch", fontsize=14, fontweight="bold")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Dice Score")
        axes[1].legend()

    plt.tight_layout()
    plt.savefig("runs/I_MMSeg_SCAR_Split/training_curves.png", dpi=300)
    plt.show()
else:
    print("Chưa tìm thấy file metrics.csv để vẽ biểu đồ.")

In [ ]:
# 2. Trực quan hóa Tốc độ Inference: Từng lát cắt, Từng ca bệnh và Bảng tổng kết
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

csv_path = Path("runs/I_MMSeg_SCAR_Split/evaluation_speed/per_case_speed_and_metrics.csv")
json_path = Path("runs/I_MMSeg_SCAR_Split/evaluation_speed/test_speed_summary.json")

if csv_path.is_file() and json_path.is_file():
    df_cases = pd.read_csv(csv_path)
    with open(json_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # A. Phân phối thời gian inference từng ca bệnh (seconds)
    sns.histplot(df_cases["case_time_sec"], kde=True, ax=axes[0], color="teal", bins=15)
    axes[0].axvline(df_cases["case_time_sec"].mean(), color="red", linestyle="--", label=f"Mean: {df_cases['case_time_sec'].mean():.2f}s")
    axes[0].set_title("Phân phối Thời gian Inference Từng Ca Bệnh (s/case)", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Thời gian (giây)")
    axes[0].set_ylabel("Số lượng ca bệnh")
    axes[0].legend()

    # B. Thời gian trung bình từng lát cắt theo ca bệnh (ms/slice)
    sns.histplot(df_cases["mean_slice_ms"], kde=True, ax=axes[1], color="coral", bins=15)
    axes[1].axvline(df_cases["mean_slice_ms"].mean(), color="blue", linestyle="--", label=f"Mean: {df_cases['mean_slice_ms'].mean():.1f}ms")
    axes[1].set_title("Phân phối Tốc độ Từng Lát Cắt (ms/slice)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Thời gian (mili-giây)")
    axes[1].set_ylabel("Tần suất")
    axes[1].legend()

    # C. Mối quan hệ giữa Số lát cắt và Thời gian xử lý ca bệnh
    sns.scatterplot(data=df_cases, x="num_slices", y="case_time_sec", ax=axes[2], color="darkviolet", s=70)
    sns.regplot(data=df_cases, x="num_slices", y="case_time_sec", ax=axes[2], scatter=False, color="indigo")
    axes[2].set_title("Tương quan: Số lát cắt vs Thời gian ca bệnh", fontsize=13, fontweight="bold")
    axes[2].set_xlabel("Số lát cắt trong Volume")
    axes[2].set_ylabel("Thời gian inference (giây)")

    plt.tight_layout()
    plt.savefig("runs/I_MMSeg_SCAR_Split/evaluation_speed/inference_speed_distribution.png", dpi=300)
    plt.show()

    # Hiển thị bảng tổng hợp đẹp mắt
    print("\n" + "="*80)
    print("⭐ BẢNG TỔNG HỢP HIỆU NĂNG TỐC ĐỘ VÀ ĐỘ CHÍNH XÁC (I_MMSeg trên 76 ca Test MyoPS-380)")
    print("="*80)
    speed_df = pd.DataFrame([{
        "Tổng số ca Test": summary["total_test_cases"],
        "Tổng số lát cắt": summary["total_test_slices"],
        "Tổng thời gian (s)": f"{summary['total_inference_time_sec']:.2f}s",
        "Tốc độ lát cắt (ms/slice)": f"{summary['slice_latency_ms']['mean']:.2f} ± {summary['slice_latency_ms']['std']:.2f} ms",
        "Tốc độ ca bệnh (s/case)": f"{summary['case_latency_sec']['mean']:.3f} s",
        "Throughput (FPS)": f"{summary['overall_throughput_fps']:.1f} fps",
        "Dice Myo": f"{summary['mean_metrics']['dice_myocardium']:.4f}",
        "Dice Scar": f"{summary['mean_metrics']['dice_scar']:.4f}",
        "Dice Edema": f"{summary['mean_metrics']['dice_edema']:.4f}",
        "Mean Dice": f"{summary['mean_metrics']['overall_mean_dice']:.4f}",
    }])
    display(speed_df)
    print("="*80)
else:
    print("Chưa tìm thấy kết quả đánh giá để trực quan hóa.")

## 💾 Bước 10: Sao lưu Checkpoints & Kết quả vào Google Drive
Đóng gói toàn bộ checkpoints, log, biểu đồ và kết quả đánh giá tốc độ inference vào Google Drive cá nhân để lưu trữ lâu dài.

In [ ]:
# Sao lưu toàn bộ thư mục runs sang Google Drive
import shutil
import os
from pathlib import Path

# Thư mục sao lưu chuẩn xác trên Google Drive (Colab Notebooks/NCKH_V2)
if os.path.exists("/content/drive/MyDrive/Colab Notebooks/NCKH_V2"):
    backup_dir = Path("/content/drive/MyDrive/Colab Notebooks/NCKH_V2/I_MMSeg_runs")
elif os.path.exists("/content/drive/MyDrive/NCKH_V2"):
    backup_dir = Path("/content/drive/MyDrive/NCKH_V2/I_MMSeg_runs")
else:
    backup_dir = Path("/content/drive/MyDrive/I_MMSeg_runs")
backup_dir.mkdir(parents=True, exist_ok=True)

run_dir = Path("runs/I_MMSeg_SCAR_Split")

if run_dir.exists():
    # 1. Nén file zip cho tiện tải về
    zip_path = "/content/I_MMSeg_SCAR_Split.zip"
    print(f"Đang nén kết quả vào {zip_path}...")
    !zip -q -r "$zip_path" "$run_dir"

    # 2. Copy zip sang Google Drive
    drive_zip = backup_dir / "I_MMSeg_SCAR_Split.zip"
    !cp "$zip_path" "$drive_zip"
    print(f"✅ Đã sao lưu file zip sang Google Drive: {drive_zip}")

    # 3. Đồng bộ trực tiếp thư mục kết quả
    !cp -r "$run_dir"/* "$backup_dir/"
    print(f"✅ Đã đồng bộ toàn bộ file sang: {backup_dir}")
    !ls -lh "$backup_dir"
else:
    print("Chưa có thư mục runs để sao lưu.")